# Sarvam-1 Quantization Benchmark — Analysis

This notebook loads `results/benchmark_results.csv` and produces:
- Quality (BLEU) vs model size tradeoff plots
- Per-language quality degradation analysis
- Speed and memory comparison across variants

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

df = pd.read_csv('../results/benchmark_results.csv')
print(f'Total rows: {len(df)}')
print(f'Variants  : {df.variant.unique()}')
print(f'Languages : {df.lang_name.unique()}')
df.head()

In [ ]:
# ── Summary table ─────────────────────────────────────────
summary = df.groupby(['variant', 'lang_name']).agg(
    avg_bleu=('bleu', 'mean'),
    em_rate=('exact_match', 'mean'),
    avg_tps=('tokens_per_sec', 'mean'),
    avg_ram_mb=('peak_ram_mb', 'mean'),
).round(2).reset_index()

summary['em_pct'] = (summary['em_rate'] * 100).round(1)
print(summary.to_string(index=False))

In [ ]:
# ── Plot 1: BLEU by variant and language ──────────────────
VARIANT_ORDER = ['fp16', 'Q8_0', 'Q4_K_M', 'Q2_K']
variants_present = [v for v in VARIANT_ORDER if v in df.variant.unique()]

langs = df.lang_name.unique()
fig, ax = plt.subplots(figsize=(10, 5))

for lang in langs:
    lang_data = summary[summary.lang_name == lang]
    lang_data = lang_data.set_index('variant').reindex(variants_present)
    ax.plot(variants_present, lang_data['avg_bleu'], marker='o', label=lang, linewidth=2)

ax.set_title('BLEU Score by Quantization Level and Language', fontsize=14)
ax.set_xlabel('Model Variant')
ax.set_ylabel('BLEU Score')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../docs/bleu_by_variant.png', dpi=150)
plt.show()

In [ ]:
# ── Plot 2: Speed vs Quality tradeoff ─────────────────────
overall = df.groupby('variant').agg(
    avg_bleu=('bleu', 'mean'),
    avg_tps=('tokens_per_sec', 'mean'),
    avg_ram_mb=('peak_ram_mb', 'mean'),
).reset_index()

fig, ax = plt.subplots(figsize=(8, 5))
for _, row in overall.iterrows():
    ax.scatter(row['avg_tps'], row['avg_bleu'], s=row['avg_ram_mb']/5, alpha=0.7, label=row['variant'])
    ax.annotate(row['variant'], (row['avg_tps'], row['avg_bleu']),
                textcoords='offset points', xytext=(8, 0), fontsize=10)

ax.set_title('Speed vs Quality (bubble size = RAM usage)', fontsize=13)
ax.set_xlabel('Tokens per second')
ax.set_ylabel('Avg BLEU')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../docs/speed_vs_quality.png', dpi=150)
plt.show()

In [ ]:
# ── Plot 3: Quality degradation relative to fp16 baseline ─
# Shows how much BLEU drops per language vs the fp16 baseline
if 'fp16' in df.variant.unique():
    baseline = summary[summary.variant == 'fp16'][['lang_name', 'avg_bleu']]
    baseline = baseline.rename(columns={'avg_bleu': 'baseline_bleu'})

    degradation = summary[summary.variant != 'fp16'].merge(baseline, on='lang_name')
    degradation['bleu_drop_pct'] = (
        (degradation['baseline_bleu'] - degradation['avg_bleu']) / degradation['baseline_bleu'] * 100
    ).round(1)

    pivot = degradation.pivot(index='variant', columns='lang_name', values='bleu_drop_pct')
    pivot = pivot.reindex([v for v in VARIANT_ORDER if v in pivot.index])

    fig, ax = plt.subplots(figsize=(10, 5))
    pivot.plot(kind='bar', ax=ax, colormap='tab10')
    ax.set_title('BLEU Degradation vs fp16 Baseline (% drop per language)', fontsize=13)
    ax.set_xlabel('Variant')
    ax.set_ylabel('BLEU Drop (%)')
    ax.yaxis.set_major_formatter(mtick.PercentFormatter())
    ax.legend(title='Language')
    ax.grid(True, alpha=0.3, axis='y')
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.savefig('../docs/bleu_degradation.png', dpi=150)
    plt.show()
else:
    print('fp16 baseline not yet available in results')